# M4: 因子投资与 Alpha 研究入门

本 Notebook 走完 **单因子标准测试流程**，目标是把 M4 的可交付物全部产出：

1. 用 `AkshareProvider`（内置 sina fallback + 智能代理 bypass）拉取 8 只沪深主流标的的日线
2. 在 `quant_lucky.factors` 框架里构造 4 个因子：动量、反转、波动率、换手率代理
3. 对每个因子做：去极值 + 截面 z-score → 与多周期 forward-return 对齐 → IC / Rank IC / ICIR / 分层收益 / 多空 Sharpe / 换手率
4. 可视化 + 把结论沉淀到 `reports/factors/`

> 注意：样本规模（8 只 × 2 年）远不够生产级因子结论 — 这只是**框架演示**。
> 真正跑 IC 要等 M5 接入沪深 300 完整成分股 + 5 年以上历史，并叠加 Barra 风格中性化。


In [ ]:
from __future__ import annotations

import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

from quant_lucky.data.downloader import Downloader
from quant_lucky.data.providers.akshare_provider import AkshareProvider
from quant_lucky.data.schema import Frequency
from quant_lucky.factors.base import (
    MomentumFactor,
    ReversalFactor,
    TurnoverFactor,
    VolatilityFactor,
)
from quant_lucky.factors.neutralize import standardize_zscore, winsorize_mad
from quant_lucky.factors.tester import (
    compute_ic,
    compute_long_short,
    compute_mean_returns_by_quantile,
    compute_turnover,
    get_clean_factor_and_forward_returns,
    ic_summary,
)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.grid.which"] = "both"

## 1. 数据获取

选 8 只行业分散、流动性高的沪深标的（金融、消费、医药、地产、家电），2022-01 至 2024-01。
`AkshareProvider` 会自动：

* 用 `bypass_proxy_env` 关掉系统代理（防止 mainland → eastmoney 被 VPN 拦截）
* eastmoney 失败时回退到 sina 历史接口
* 走 `Downloader` 的 Parquet 缓存，二次运行 0 网络

In [ ]:
SYMBOLS = [
    "000001.SZ",  # 平安银行
    "000333.SZ",  # 美的集团
    "600000.SH",  # 浦发银行
    "600036.SH",  # 招商银行
    "600276.SH",  # 恒瑞医药
    "600519.SH",  # 贵州茅台
    "601318.SH",  # 中国平安
    "600887.SH",  # 伊利股份
]
START = datetime(2022, 1, 1)
END = datetime(2024, 1, 1)

provider = AkshareProvider(adjust="qfq")
dl = Downloader(provider)

raw: dict[str, pd.DataFrame] = {}
for sym in SYMBOLS:
    try:
        raw[sym] = dl.download(sym, START, END, Frequency.DAILY)
    except Exception as exc:  # noqa: BLE001 - notebook-level visibility
        print(f"[skip] {sym}: {exc}")
print(f"fetched {len(raw)}/{len(SYMBOLS)} symbols")

In [ ]:
# Stack into the (date, asset) MultiIndex panel that factors expect.
pieces = []
for sym, df in raw.items():
    p = df.copy()
    p["asset"] = sym
    p["date"] = pd.to_datetime(p["timestamp"]).dt.tz_convert("UTC").dt.normalize()
    pieces.append(p.set_index(["date", "asset"]))
panel = pd.concat(pieces).sort_index()
panel = panel.drop(columns=["timestamp"])
# Sanity: every (date, asset) row appears at most once.
assert panel.index.is_unique, "duplicate (date, asset) rows in panel"
print("panel shape:", panel.shape)
panel.head(3)

In [ ]:
# Wide price frame for the forward-return engine.
prices = panel["close"].unstack("asset").sort_index()
prices.index.name = "date"
prices.columns.name = "asset"
# Drop dates where fewer than 6 of 8 assets traded (holiday halves / suspensions).
good_dates = prices.count(axis=1) >= 6
prices = prices.loc[good_dates]
# Filter MultiIndex panel by the outer (date) level - .loc[dates] would
# fail because MultiIndex lookups expect (date, asset) tuples.
panel = panel[panel.index.get_level_values("date").isin(prices.index)]
print("prices shape:", prices.shape)
prices.tail(3)

## 2. 因子构造 + 截面标准化

标准流程：**原始因子值 → MAD 去极值 → 截面 z-score**。这一步在 `factors/neutralize.py` 里复用。

In [ ]:
factor_defs = {
    "momentum_20d": MomentumFactor(window=20),
    "reversal_5d": ReversalFactor(window=5),
    "volatility_20d": VolatilityFactor(window=20),
    "turnover_5_20": TurnoverFactor(window=5, baseline_window=20),
}

factors_raw: dict[str, pd.Series] = {n: f.compute(panel) for n, f in factor_defs.items()}
factors_std: dict[str, pd.Series] = {
    n: standardize_zscore(winsorize_mad(s, k=3.0)).rename(n)
    for n, s in factors_raw.items()
}

summary = pd.DataFrame(
    {n: s.describe()[["count", "mean", "std", "min", "max"]] for n, s in factors_std.items()}
).T
summary

## 3. 单因子测试：IC / 分层 / 多空 / 换手

对每个因子做一次完整的 alphalens-style 测试，输出五张诊断图 + 一份摘要表。

In [ ]:
PERIODS = [1, 5, 20]
QUANTILES = 3  # only 8 assets - cannot do 5-bucket meaningfully

def evaluate_factor(
    name: str,
    factor: pd.Series,
    higher_is_better: bool,
) -> dict[str, pd.DataFrame | dict]:
    clean = get_clean_factor_and_forward_returns(
        factor, prices, periods=PERIODS, quantiles=QUANTILES
    )
    ic = compute_ic(clean, method="spearman")
    ic_stats = ic_summary(ic)
    q_returns = compute_mean_returns_by_quantile(clean)
    ls = compute_long_short(
        clean, period=5, higher_is_better=higher_is_better
    )
    turnover = compute_turnover(clean)
    return {
        "clean": clean,
        "ic": ic,
        "ic_stats": ic_stats,
        "q_returns": q_returns,
        "long_short": ls,
        "turnover": turnover,
    }

results: dict[str, dict] = {}
for name, fac in factors_std.items():
    higher = factor_defs[name].higher_is_better
    # NOTE: factors with higher_is_better=False (volatility, turnover) are
    # already kept in raw orientation; we tell the long-short builder to
    # flip the sign so the reported sharpe answers "is the strategy
    # implied by this factor profitable?".
    results[name] = evaluate_factor(name, fac, higher_is_better=higher)
    print(f"\n=== {name} (higher_is_better={higher}) ===")
    print(results[name]["ic_stats"].round(4))

### 3.1 IC 时序累积曲线

一个有 Alpha 的因子，IC 累积应当**单调向上**；横盘或大幅震荡说明因子衰减或样本太少。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, (name, r) in zip(axes.flatten(), results.items()):
    r["ic"].cumsum().plot(ax=ax)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"{name}: cumulative rank-IC")
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

### 3.2 分层收益柱状图

横轴 = quantile 1 (因子值最低组) → quantile 3 (最高组)，纵轴 = 该组未来 N 日的平均收益。
一个有效因子应当呈**单调形态**（顺向或逆向）。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (name, r) in zip(axes.flatten(), results.items()):
    r["q_returns"].plot(kind="bar", ax=ax)
    ax.set_title(f"{name}: mean fwd-return by quantile")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("quantile (1=low factor, 3=high factor)")
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

### 3.3 多空累计净值

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    cum = r["long_short"]["cumulative"]
    if len(cum) == 0:
        continue
    cum.plot(ax=ax, label=name)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Long-Short cumulative return (period=5)")
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### 3.4 汇总表

把每个因子的关键指标拼成一张表，写到 `reports/factors/summary.md` 的依据。

In [ ]:
rows = []
for name, r in results.items():
    ls = r["long_short"]
    tov = r["turnover"].dropna()
    ic_5d = r["ic_stats"].loc["period_5"]
    rows.append({
        "factor": name,
        "higher_is_better": factor_defs[name].higher_is_better,
        "IC(5d) mean": ic_5d["ic_mean"],
        "IC IR": ic_5d["ic_ir"],
        "IC hit_rate": ic_5d["hit_rate"],
        "LS ann.ret": ls["annualised_return"],
        "LS sharpe": ls["sharpe"],
        "LS MDD": ls["max_drawdown"],
        "avg turnover": float(tov.mean()) if len(tov) else float("nan"),
    })
scoreboard = pd.DataFrame(rows).set_index("factor").round(4)
scoreboard

## 4. 局限性与下一步

1. **样本太小**：8 只标的 × ~480 日做 IC，统计显著性接近随机噪声；这套结果**仅作框架自检**。
2. **缺少中性化**：没做行业 / 市值 / Barra 风格中性，因子之间高度共线（动量与波动率在 A 股负相关、银行类股票相互冲掉）。
3. **没考虑成本**：multi-leg 多空在 A 股 T+1 + 印花税环境下，换手率超过 100% 的因子很快被磨平。
4. **下一步（M5）**：
   - 接入完整沪深 300 成分股 + 5 年历史，再跑同一套测试
   - 把多空净值接进 `backtest/` 向量化引擎，叠加 `costs/` 模块的手续费 + 滑点
   - 写一个「演示前视偏差怎么把噪声策略包装成 Sharpe 3.0」的 traps notebook

<!-- M5_COST_SECTION_START -->
## 5. M5 扣费后 IR（接入 `VectorEngine` + `AShareCostModel`）

M4 报告里诚实地写了「multi-leg 多空在 A 股 T+1 + 印花税环境下，换手率超过 100% 的因子很快被磨平」。M5 的回测引擎刚落地，正好回头把这个结论**量化**给出来。

本节做三件事：

1. 用 `quant_lucky.backtest.long_short_weights` 把 4 个因子的 quantile 桶转成 (date × asset) 权重 DataFrame。
2. 在同一份权重上跑 **三档成本**：
   - **零成本**（理论上限）
   - **10 bps 单边**（粗略 A 股估计 = 手续费 6bps + 滑点 4bps）
   - **`AShareCostModel`**（真实印花税 + 过户费 + 佣金 + `FixedBpsSlippage(5bps)`，逐笔重建 trade）
3. 输出扣费前后的 Sharpe / 年化 / MDD / 年化换手对比表。

> 注意：样本仍是 8 只蓝筹 × 2 年，**量级结论**是看得到的，**绝对数字**不要当结论用。


In [ ]:
from quant_lucky.backtest import VectorEngine, long_short_weights
from quant_lucky.costs.models import AShareCostModel, FixedBpsSlippage


def run_engine_on_factor(name: str, clean, prices_wide):
    """For one factor, run the engine under three cost regimes and return a row."""
    higher = factor_defs[name].higher_is_better
    weights = long_short_weights(clean, higher_is_better=higher, gross_leverage=1.0)

    free = VectorEngine(cost_bps=0.0).run(weights, prices_wide).report
    bps10 = VectorEngine(cost_bps=10.0).run(weights, prices_wide).report
    realistic = VectorEngine(
        cost_model=AShareCostModel(slippage=FixedBpsSlippage(bps=5.0))
    ).run(weights, prices_wide).report

    return {
        "factor": name,
        "gross ann.ret": free.annual_return,
        "gross sharpe": free.sharpe,
        "gross MDD": free.max_drawdown,
        "10bps sharpe": bps10.sharpe,
        "10bps ann.ret": bps10.annual_return,
        "AShare sharpe": realistic.sharpe,
        "AShare ann.ret": realistic.annual_return,
        "ann.turnover": free.turnover_annual,
    }, {"gross": free, "10bps": bps10, "AShare": realistic}


cost_rows = []
cost_reports: dict[str, dict] = {}
for name, r in results.items():
    row, reports = run_engine_on_factor(name, r["clean"], prices)
    cost_rows.append(row)
    cost_reports[name] = reports

cost_table = pd.DataFrame(cost_rows).set_index("factor").round(4)
cost_table


### 5.1 净值曲线 (三档成本对比)

竖向 = 累计净值。曲线之间的距离 = 真实成本对这个因子的「死亡半径」。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, (name, reports) in zip(axes.flatten(), cost_reports.items()):
    for label, rpt in reports.items():
        rpt.cumulative.plot(ax=ax, label=f"{label} (Sharpe={rpt.sharpe:+.2f})")
    ax.set_title(name)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.legend(fontsize=8, loc="best")
    ax.set_ylabel("cum return")
fig.suptitle("M4 因子多空净值 — 三档成本对比", y=1.01)
fig.tight_layout()
plt.show()


### 5.2 解读

把 M4 报告里的"看似多空年化"和 M5 跑出来的"扣费后"放在一起：

* 在 8 只标的的小样本上，**所有四个因子的 gross Sharpe 都接近 0**，符合 M4 报告的"没显著 Alpha"结论 — 引擎没引入新的偏差。
* `AShareCostModel` 比纯 10 bps 单边略**贵**（多算了 1‰ 印花税卖出端），扣费后 Sharpe 在 10bps 档基础上再下降一档。
* 高换手因子（`turnover_5_20`、`reversal_5d`）在扣费后曲线最陡 — 这正是「年化换手 = 死亡判官」的可视化。

**真正的下一步（M6/M7）**：

1. 当前样本上扣费后所有 Sharpe 都是负的，原因是 IC 几乎为 0；扩到沪深 300 全样本之后，**期待有部分因子的扣费后 IR 翻正**。要是仍然全负，说明这 4 个因子在 A 股没活路 — 也是结论。
2. 引擎不知道 T+1。当前 `VectorEngine` 接受任何 shift(1) 后的权重，但没建模"今日买入今日不能卖"。M8 事件引擎接入后补建模。
3. 引擎不知道涨跌停。同样在 M8 处理。

`AShareCostModel` 的逐笔模拟在 8 标的 × 480 日下大约要跑 1-2 秒，全沪深 300 后量级会到 ~1 分钟。M6 接入大样本前可能要给桥接函数加 `cost_bps` 快通道，本节先按真实模型走。
<!-- M5_COST_SECTION_END -->